In [ ]:
import pandas as pd
import numpy as np

import re
import string
import joblib
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

In [ ]:
import sys

print(sys.executable)

In [ ]:
# Download required NLTK resources

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
train_df = pd.read_csv(
    "../data/train_data.txt",
    sep=" ::: ",
    engine="python",
    header=None,
    names=["ID", "Title", "Genre", "Description"]
)

In [ ]:
test_df = pd.read_csv(
    "../data/test_data.txt",
    sep=" ::: ",
    engine="python",
    header=None,
    names=["ID", "Title", "Description"]
)

In [ ]:
test_solution = pd.read_csv(
    "../data/test_data_solution.txt",
    sep=" ::: ",
    engine="python",
    header=None,
    names=["ID", "Title", "Genre", "Description"]
)

In [ ]:
print("Train Shape :", train_df.shape)
print("Test Shape :", test_df.shape)
print("Test Labels Shape :", test_solution.shape)

In [ ]:
# Initialize stopwords and lemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

print("Total Stopwords:", len(stop_words))

In [ ]:
# Text preprocessing function

def preprocess_text(text):
    
    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [word for word in words if word not in stop_words]

    # Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [ ]:
sample_text = train_df.loc[0, "Description"]

print("Original Text:\n")
print(sample_text)

print("\n" + "="*100 + "\n")

print("Clean Text:\n")
print(preprocess_text(sample_text))

In [ ]:
# Apply preprocessing to training and test datasets

train_df["clean_description"] = train_df["Description"].apply(preprocess_text)

test_df["clean_description"] = test_df["Description"].apply(preprocess_text)

print("Preprocessing completed successfully!")

In [ ]:
# Convert text into TF-IDF features

tfidf = TfidfVectorizer(
    max_features=5000
)

X_train = tfidf.fit_transform(train_df["clean_description"])
X_test = tfidf.transform(test_df["clean_description"])

y_train = train_df["Genre"]
y_test = test_solution["Genre"]

In [ ]:
print("X_train Shape :", X_train.shape)
print("X_test Shape  :", X_test.shape)

print("y_train Shape :", y_train.shape)
print("y_test Shape  :", y_test.shape)

In [ ]:
# Train Multinomial Naive Bayes

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

print("Multinomial Naive Bayes model trained successfully!")

In [ ]:
# Predict on test data

nb_predictions = nb_model.predict(X_test)

In [ ]:
# Evaluate Naive Bayes model

nb_accuracy = accuracy_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions, average="macro")

print(f"Accuracy : {nb_accuracy:.4f}")
print(f"Macro F1 Score : {nb_f1:.4f}")

In [ ]:
# Classification Report

print(classification_report(y_test, nb_predictions))

In [ ]:
# Train Logistic Regression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

In [ ]:
# Predict on test data

lr_predictions = lr_model.predict(X_test)

In [ ]:
# Evaluate Logistic Regression

lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_f1 = f1_score(y_test, lr_predictions, average="macro")

print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Macro F1 Score : {lr_f1:.4f}")

In [ ]:
print(classification_report(y_test, lr_predictions))

In [ ]:
# Train Linear Support Vector Machine

svm_model = LinearSVC(random_state=42)

svm_model.fit(X_train, y_train)

print("Linear SVM model trained successfully!")

In [ ]:
# Predict on test data

svm_predictions = svm_model.predict(X_test)

In [ ]:
# Evaluate Linear SVM

svm_accuracy = accuracy_score(y_test, svm_predictions)
svm_f1 = f1_score(y_test, svm_predictions, average="macro")

print(f"Accuracy : {svm_accuracy:.4f}")
print(f"Macro F1 Score : {svm_f1:.4f}")

In [ ]:
print(classification_report(y_test, svm_predictions))

In [ ]:
# Compare model performance

comparison = pd.DataFrame({
    "Model": [
        "Multinomial Naive Bayes",
        "Logistic Regression",
        "Linear SVM"
    ],
    "Accuracy": [
        nb_accuracy,
        lr_accuracy,
        svm_accuracy
    ],
    "Macro F1 Score": [
        nb_f1,
        lr_f1,
        svm_f1
    ]
})

comparison = comparison.sort_values(by="Accuracy", ascending=False)

comparison

In [ ]:
# Save Logistic Regression model and TF-IDF vectorizer

joblib.dump(lr_model, "../models/best_model.joblib")
joblib.dump(tfidf, "../models/tfidf_vectorizer.joblib")

print("Model and TF-IDF Vectorizer saved successfully!")

In [ ]:
# # Conclusion

# Three machine learning models were trained and evaluated for movie genre classification.

# - Multinomial Naive Bayes served as the baseline model.
# - Logistic Regression achieved the highest overall accuracy.
# - Linear SVM obtained the highest Macro F1 score.

# Based on overall performance, Logistic Regression was selected as the final model for deployment.

# The trained model and TF-IDF vectorizer were saved for use in the Streamlit web application.